# Exploring the data

Use the **Python (adatl1)** kernel for this notebook. See `setup.ipynb` if it's missing.

<!-- TODO(organisers): physics introduction. What is an "event"? What are L1 trigger objects? -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak

import dataset

## 1. Loading events

`dataset.load_events` reads parquet files into an [awkward array](https://awkward-array.org/), which handles
**jagged** data: every event has a different number of jets, muons, and so on.

Pass `max_events` to load only part of a large sample while you explore.

In [ ]:
zb = dataset.load_events("ZB_run396102", "train", max_events=200_000, physical_units=True)
sig = dataset.load_events("GluGluHto2Tau_Par-MH-125", "test", max_events=200_000, physical_units=True)

print(f"zero bias: {len(zb):,} events | signal: {len(sig):,} events")
print(zb.fields)

In [ ]:
# One event, as a Python dict
zb[0].to_list()

## 2. Object multiplicities

<!-- TODO(organisers): discussion prompt. Why might signal events have more / harder objects? -->

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, coll in zip(axes, dataset.OBJECTS):
    bins = np.arange(0, 14) - 0.5
    for name, ev in [("zero bias", zb), ("signal", sig)]:
        ax.hist(ak.num(ev[coll]), bins=bins, histtype="step", density=True, label=name)
    ax.set_xlabel(f"number of {coll}")
axes[0].legend()
plt.tight_layout()

## 3. Kinematic distributions

With `physical_units=True`, Et is in GeV, eta is pseudorapidity, and phi is in radians.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
coll = "jets"  # try "muons", "egammas", "taus"
for ax, feat, bins in zip(axes, ["Et", "eta", "phi"],
                          [np.linspace(0, 300, 61), np.linspace(-5, 5, 51), np.linspace(0, 2 * np.pi, 49)]):
    for name, ev in [("zero bias", zb), ("signal", sig)]:
        ax.hist(ak.flatten(ev[coll][feat]), bins=bins, histtype="step", density=True, label=name)
    ax.set_xlabel(f"{coll} {feat}")
axes[0].set_yscale("log")
axes[0].legend()
plt.tight_layout()

In [ ]:
# Energy sums have exactly one entry per event
bins = np.linspace(0, 500, 51)
for name, ev in [("zero bias", zb), ("signal", sig)]:
    plt.hist(ak.flatten(ev.HT.Et), bins=bins, histtype="step", density=True, label=name)
plt.yscale("log"); plt.xlabel("HT [GeV]"); plt.legend();

## 4. The existing trigger decision

`L1bit` is `True` when the **standard** L1 trigger menu accepted the event. `dataset.load_seeds(...)` gives
the individual algorithm decisions, which you can use to benchmark your anomaly detector.

<!-- TODO(organisers): explain trigger rates and the bandwidth budget. -->

In [ ]:
print(f"Fraction of zero-bias events accepted by the menu: {ak.mean(zb.L1bit):.4f}")
print(f"Fraction of signal events accepted by the menu:    {ak.mean(sig.L1bit):.4f}")

## 5. From jagged events to ML-ready arrays

Most neural networks want fixed-size inputs. `dataset.to_padded` keeps the highest-Et objects from each
collection, pads the rest with zeros, and returns a mask of which slots are real.

The default layout is 1 MET + 4 muons + 10 jets + 12 e-gammas + 12 taus = **39 slots × 3 features (Et, eta, phi)**.
In the training scripts, the layout is set in the `input:` section of the config files in `config/`.

In [ ]:
print(dataset.DEFAULT_LAYOUT)
x, mask = dataset.to_padded(zb)
print("x:", x.shape, x.dtype, "| mask:", mask.shape)

In [ ]:
# A custom layout
x_small, _ = dataset.to_padded(zb, layout={"MET": 1, "jets": 6, "muons": 2}, features=("Et", "eta", "phi"))
print(x_small.shape)

## 6. Your turn

<!-- TODO(organisers): exercises. e.g.
 - Compare different signal samples: which look most different from zero bias?
 - Compare zero-bias data with the SingleNeutrino simulation.
-->

When you're ready, continue with [`train.ipynb`](train.ipynb).